[![Open In Colab](https://google.com)](https://google.com)

# Open_E-Dope APK Compiler Notebook
This notebook automates the system dependency installation, repository fetching, patch processing, and Buildozer compilation for the **Open_E-Dope** application.

### Step 1: Install Ubuntu System Dependencies
Buildozer requires standard Linux development headers, utilities, and **Java 17** to cross-compile the Android toolchain.

In [ ]:
# Update apt registries
!sudo apt-get update

# Install essential packages and Java 17
!sudo apt-get install -y \
    build-essential \
    uuid-dev \
    zlib1g-dev \
    libncurses5-dev \
    libgdbm-dev \
    libnss3-dev \
    libssl-dev \
    libsqlite3-dev \
    libreadline-dev \
    libffi-dev \
    libbz2-dev \
    liblzma-dev \
    tk-dev \
    libgdbm-compat-dev \
    gawk \
    unzip \
    zip \
    autoconf \
    automake \
    libtool \
    pkg-config \
    cmake \
    git \
    openjdk-17-jdk

# Verify Java 17 runtime environmental binding
!java -version

### Step 2: Install Buildozer & Pin Cython
Colab updates break older environments by shipping newer version iterations of Cython. Pinning `cython==0.29.36` keeps things stable for Kivy compilation builds.

In [ ]:
!pip install --upgrade buildozer cython==0.29.36

### Step 3: Clone Open_E-Dope Repository
This pulls down the project layout structure along with your pre-configured `buildozer.spec` manifest file.

In [ ]:
import os

if not os.path.exists('Open_E-Dope'):
    !git clone https://github.com

# Change path context to the target repository directory root
%cd Open_E-Dope

### Step 4: Patch Buildozer Configuration
This steps overrides the default Python version configuration inside your `buildozer.spec` script, forcing it to compile using stable **Python 3.11** recipes instead of falling forward to unsupported Python 3.14 pre-releases.

In [ ]:
import re

with open('buildozer.spec', 'r') as file:
    content = file.read()

# Remove any existing commented or active python3 requirements overrides
content = re.sub(r'#?\s*python3\s*==\s*[\d\.]+', '', content)

# Ensure requirements contains python3==3.11
if "requirements =" in content:
    content = content.replace("requirements = ", "requirements = python3==3.11, ")

with open('buildozer.spec', 'w') as file:
    file.write(content)

print("Successfully patched buildozer.spec to target Python 3.11!")

### Step 5: Run Buildozer Compilation
This processes the local `buildozer.spec` script. It downloads the Android SDK/NDK, processes your custom Java source files (`NfcHelper.java`), and packages everything into an installable application bundle.

**Note:** Keep your eye on this cell as it runs—you will need to type `y` and hit **Enter** to accept the Android SDK license terms midway through the run.

In [ ]:
# Clean up any corrupted previous build attempts first, then compile
!buildozer android clean
!buildozer -v android debug

### Step 6: Locate your APK
Once compilation completes successfully, run this cell to verify your output binary inside the `bin/` subfolder.

In [ ]:
!ls -la bin/